# CT-Explain — End-to-end demo

This notebook exercises every component of the paper on a small
synthetic graph, so it runs on a plain laptop with no GPU or Kaggle
credentials. Replace the `synthetic=True` switches with the real dataset
paths once you have the Kaggle bundles downloaded.

Sections:
1. Build a continuous-time graph
2. CT-TGNN / SDE-TGNN detection
3. Four explanations + calibrated intervals
4. ConformalGuard calibration, certification, anytime-valid monitoring
5. SOC dashboard JSON contract


In [ ]:
import torch
from ct_explain.utils.seed import set_global_seed
set_global_seed(0)

from ct_explain.data.datasets import GeneralNetworkTrafficDataset
ds = GeneralNetworkTrafficDataset(synthetic=True, synthetic_rows=1024)
df = ds.load_training_frame()
graph = ds.build_graph(df)
print('nodes:', graph.num_nodes, 'edges:', graph.num_edges)

In [ ]:
from ct_explain.models.sde_tgnn import SDETemporalGNN

model = SDETemporalGNN(
    node_feat_dim=graph.node_features.shape[1],
    edge_feat_dim=graph.edge_features.shape[1],
    hidden_dim=32, num_classes=2,
    num_heads=2, time_dim=16, ode_solver='euler', adjoint=False,
).eval()
out = model.predict(
    graph.node_features, graph.edge_index,
    graph.edge_features, graph.edge_times,
)
print('softmax[0]:', out['probabilities'][0].tolist())

In [ ]:
from ct_explain.soc.human_ai import HumanAICollaboration
collab = HumanAICollaboration(model, class_names=['safe', 'attack'])
alert = collab.triage(graph=graph, target_node=0,
                      alert_id='demo_alert_0', source='notebook')
panel = collab.investigate(graph=graph, alert=alert)
list(panel.keys())

In [ ]:
from ct_explain.conformal.conformal_guard import ConformalGuard
from ct_explain.conformal.execution_graph import DynamicExecutionGraph, NodeKind, EdgeKind

guard = ConformalGuard(alpha=0.1, k_hops=1, lambda_cascade=0.0)

calib = []
for i in range(120):
    g = DynamicExecutionGraph()
    g.add_node(f'a_{i}', NodeKind.ACTION, timestamp=i, attrs={})
    calib.append((f'a_{i}', g, (lambda: (lambda _: 0.9))(), 1))
q_hat = guard.calibrate(calib)
print('q_hat:', q_hat)
print(guard.certify('a_0', calib[0][1], lambda _: 0.95))